# 01 — Exploratory Data Analysis, Quirk Cleaning & Demographic Base Rates

**Objective:** Ingest the UCI *Default of Credit Card Clients* dataset ($N=30,000$), document and clean schema quirks (`PAY_0`, undocumented `EDUCATION` and `MARRIAGE` codes), derive `AGE_GROUP` bins, verify zero data leakage, and inspect base default rates by protected group.


In [1]:
import json
import pandas as pd
from creditfair import config
from creditfair.data import load_saved_splits

splits = load_saved_splits()
print('Split sizes:', splits.cleaning_report['split_sizes'])
print('Overall default rate:', f"{splits.cleaning_report['default_rate']:.2%}")
print('Undocumented EDUCATION rows remapped to 4:', splits.cleaning_report['education_undocumented_remapped_to_4'])
print('Undocumented MARRIAGE rows remapped to 3:', splits.cleaning_report['marriage_undocumented_remapped_to_3'])


Split sizes: {'train': 21000, 'val': 4500, 'test': 4500}
Overall default rate: 22.12%
Undocumented EDUCATION rows remapped to 4: 345
Undocumented MARRIAGE rows remapped to 3: 54


### Analysis of Dataset Hygiene & Quirks
- Out of **30,000 rows**, **345 rows** contained undocumented `EDUCATION` codes (`0, 5, 6`) and **54 rows** contained `MARRIAGE = 0`. Rather than dropping these applicants, we consolidate them into the documented *Others* category (`4` and `3` respectively).
- `PAY_0` has been renamed to `PAY_1` so the 6-month repayment status sequence (`PAY_1`..`PAY_6`) aligns with `BILL_AMT1`..`BILL_AMT6` and `PAY_AMT1`..`PAY_AMT6`.


In [1]:
# Inspect demographic base default rates across SEX, AGE_GROUP, and MARRIAGE
for attr, group_dict in splits.cleaning_report['group_base_rates'].items():
    print(f'=== {attr} ===')
    print(pd.DataFrame(group_dict).T)


{
  "SEX": {
    "Female": {
      "n": 18112,
      "default_rate": 0.2078
    },
    "Male": {
      "n": 11888,
      "default_rate": 0.2417
    }
  },
  "AGE_GROUP": {
    "21-29": {
      "n": 9618,
      "default_rate": 0.2284
    },
    "30-39": {
      "n": 11238,
      "default_rate": 0.2025
    },
    "40-49": {
      "n": 6464,
      "default_rate": 0.2297
    },
    "50+": {
      "n": 2680,
      "default_rate": 0.253
    }
  },
  "MARRIAGE": {
    "Married": {
      "n": 13659,
      "default_rate": 0.2347
    },
    "Others": {
      "n": 377,
      "default_rate": 0.2361
    },
    "Single": {
      "n": 15964,
      "default_rate": 0.2093
    }
  },
  "SEX_x_AGE_GROUP": {
    "Female | 21-29": {
      "n": 6337,
      "default_rate": 0.2222
    },
    "Female | 30-39": {
      "n": 6673,
      "default_rate": 0.1837
    },
    "Female | 40-49": {
      "n": 3693,
      "default_rate": 0.215
    },
    "Female | 50+": {
      "n": 1409,
      "default_rate": 0.2378
    

### Conclusion: Why Group Base Rates Matter for Fairness
- Male applicants exhibit a higher base default rate (**24.17%**) than Female applicants (**20.78%**).
- Across age brackets, `21-29` (**22.84%**) and `50+` (**25.30%**) have higher default rates than prime working-age `30-39` borrowers (**20.25%**).
- By the **Impossibility Theorem of Fairness**, whenever base rates differ across protected groups, no calibrated classifier can simultaneously achieve equal selection rates (Demographic Parity) and equal error rates (Equalized Odds).
